In [33]:
import osmnx as ox
import networkx as nx
import numpy as np
import json
import tqdm
from rich.console import Console


def has_signal(city, node):
        attrs = city.nodes[node]
        if not 'highway' in attrs:
            return False
        elif attrs['highway'] == "traffic_signals":
            return True
        else:
            return False

def get_num_signals(city, nodes_visited):
    num_signals = sum([has_signal(city, p) for p in nodes_visited])
    return num_signals


console = Console()

with console.status("[bold green]Loading computational parameters...") as status:
    console.log("Loading city data...")
    centroids = json.load(open("../data/tract-centroids.json"))
    pairs = json.load(open("../data/origin_pairs.json"))
    console.log("Finished loading city data!")

    ## IMPORT CITY
    console.log("Loading road network data (takes a while)...")
    city = ox.graph_from_place("Chicago, IL, USA", network_type="drive")

    ## ADDING EDGE PROPERTIES
    city = ox.add_edge_speeds(city)
    city = ox.add_edge_travel_times(city)
    console.log("Finished loading road network data!")

    ## SNAPPING TRACT CENTROIDS TO THE ROAD NETWORK
    all_tracts = list(centroids)
    node_ids = ox.nearest_nodes(city,
                                [centroids[t][0] for t in all_tracts],
                                [centroids[t][1] for t in all_tracts])
    tract_nodes = {t: int(n) for t, n in zip(all_tracts, node_ids)}

[14:01:17] Loading city data...                                                                    ]8;id=2935492;file:///var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_60687/3254546517.py\3254546517.py]8;;\:]8;id=2935493;file:///var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_60687/3254546517.py#26\26]8;;\

           Finished loading city data!                                                             ]8;id=2935498;file:///var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_60687/3254546517.py\3254546517.py]8;;\:]8;id=2935499;file:///var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_60687/3254546517.py#29\29]8;;\

           Loading road network data (takes a while)...                                            ]8;id=2935504;file:///var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_60687/3254546517.py\3254546517.py]8;;\:]8;id=2935505;file:///var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_60687/3254546517.py#32\32]8;;\

/Users/jcrb/.julia/conda/3/aarch64/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install "ipywidgets"
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/jcrb/.julia/conda/3/aarch64/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install "ipywidgets"
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

/Users/jcrb/.julia/conda/3/aarch64/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install "ipywidgets"
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/jcrb/.julia/conda/3/aarch64/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install "ipywidgets"
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/jcrb/.julia/conda/3/aarch64/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install "ipywidgets"
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[14:01:35] Finished loading road network data!                                                     ]8;id=2935510;file:///var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_60687/3254546517.py\3254546517.py]8;;\:]8;id=2935511;file:///var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_60687/3254546517.py#38\38]8;;\

/Users/jcrb/.julia/conda/3/aarch64/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install "ipywidgets"
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [38]:
ox.save_graphml(city, filepath='../data/chicago.graphml')

## Computing which nodes have signals

In [22]:
output_file = "../data/signal_nodes.json"
results = {}

for node in tqdm.tqdm(city.nodes):
   results[node] = has_signal(city, node)

json.dump(results, open(output_file, "w"), indent=2)
console.log(f"Saved node signal data to {output_file}")

100%|██████████| 29807/29807 [00:00<00:00, 1456137.76it/s]


[11:54:25] Saved node signal data to ../data/signal_nodes.json                                      ]8;id=2935486;file:///var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_60687/1412111479.py\1412111479.py]8;;\:]8;id=2935487;file:///var/folders/x_/mtr88x3523j2pp7lgmpl_7k40000gp/T/ipykernel_60687/1412111479.py#8\8]8;;\

## Computing signals for origin/destination paths

In [ ]:
output_file = "../data/num_traffic_signals.json"
results = {}

for tract in tqdm.tqdm(pairs):
    if tract not in tract_nodes:
        continue
    # one Dijkstra per origin gives the shortest (travel time) path to every destination
    paths = nx.single_source_dijkstra_path(city, tract_nodes[tract], weight="travel_time")
    for dest in np.unique(pairs[tract]):
        dest = str(dest)
        if dest not in tract_nodes or tract_nodes[dest] not in paths:
            continue
        num_signals = get_num_signals(city, paths[tract_nodes[dest]])
        results.setdefault(tract, {})[dest] = int(num_signals)

json.dump(results, open(output_file, "w"), indent=2)
console.log(f"Saved traffic signal counts to {output_file}")

In [ ]:
indexes = {node: i for i, node in enumerate(city.nodes)}
signal_dict = json.load(open('../data/signal_nodes.json', 'r'))

c = 26*np.array([signal_dict[str(node)] for node, idx in indexes.items()])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 26, 0, 0, 26, 0, 0, 26, 0, 0, 26, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 26, 26, 26, 26, 26, 26, 26, 0, 26, 0, 26, 0, 26, 0, 0, 0, 0, 26, 0, 0, 0, 26, 0, 0, 26, 26, 0, 26, 0, 0, 26, 0, 0, 0, 0, 0, 0, 0, 0, 0, 26, 26, 0, 0, 26, 26, 26, 0, 0, 0, 0, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 0, 0, 26, 26, 0, 26, 26, 26, 26, 26, 26, 0, 0, 26, 0, 26, 26, 26, 0, 0, 26, 26, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 26, 0, 26, 0, 0, 0, 0, 0, 0, 0, 26, 0, 26, 26, 0, 0, 0, 0, 0, 0, 0, 0, 0, 26, 26, 26, 26, 26, 26, 26, 0, 26, 26, 26, 26, 26, 26, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 0, 0, 26, 26, 26, 26, 0, 26, 26, 26, 26, 26, 26, 26, 26, 0, 0, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 2